In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
Folder_path="/content/drive/MyDrive/Calendar-Assistant NLU"
train_path = os.path.join(Folder_path, "train.json")
val_path   = os.path.join(Folder_path, "val.json")
test_path  = os.path.join(Folder_path, "test.json")

In [3]:
import json
import numpy as np
train_data=json.load(open(train_path))
val_data=json.load(open(val_path))
test_data=json.load(open(test_path))
print(train_data[0])

{'raw_text': 'set a reminder to call emma at midnight', 'tokens': ['set', 'a', 'reminder', 'to', 'call', 'emma', 'at', 'midnight'], 'tags': ['O', 'O', 'O', 'O', 'O', 'B-PERSON', 'O', 'B-TIME'], 'intent': 'SET_REMINDER', 'date_iso': None, 'time_hm': '00:00', 'id': 'ex_002698', 'target_string': 'SET_REMINDER|O O O O O B-PERSON O B-TIME|00:00'}


In [4]:
import torch
!pip install -q gensim
import gensim.downloader as api
fasttext = api.load("fasttext-wiki-news-subwords-300")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.2 MB/s eta 0:00:00
[==================================================] 100.0% 958.5/958.4MB downloaded


In [5]:
from collections import Counter

counter = Counter()

for sample in train_data:
    counter.update(sample["tokens"])

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

print("Vocabulary Size:", len(word2idx))

Vocabulary Size: 649


In [6]:
embedding_dim=300
embedding_matrix=np.random.normal(size=(len(word2idx),embedding_dim))
for word in word2idx:
    if word in fasttext:
        embedding_matrix[word2idx[word]]=fasttext[word]

In [7]:
import torch.nn as nn
embedding=nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix),freeze=False)

In [8]:
tag2idx={"<pad>":0}
for sample in train_data:
  for tag in sample["tags"]:
    if tag not in tag2idx :
      tag2idx[tag]=len(tag2idx)
idx2tag={idx:word for word,idx in tag2idx.items()}
print(tag2idx)

{'<pad>': 0, 'O': 1, 'B-PERSON': 2, 'B-TIME': 3, 'B-EVENT': 4, 'I-EVENT': 5, 'B-DATE': 6, 'I-DATE': 7}


In [9]:
X_train = []
Y_train = []
for sample in train_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_train.append(sentence)
    Y_train.append(tags)
X_test = []
Y_test = []
for sample in test_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_test.append(sentence)
    Y_test.append(tags)
X_val = []
Y_val= []
for sample in val_data:
    sentence = [
        word2idx.get(word, word2idx["<UNK>"])
        for word in sample["tokens"]
    ]

    tags = [
        tag2idx[tag]
        for tag in sample["tags"]
    ]

    X_val.append(sentence)
    Y_val.append(tags)

In [10]:
maxlen1=max(len(sentence) for sentence in X_train)
maxlen2=max(len(sentence) for sentence in X_test)
maxlen3=max(len(sentence) for sentence in X_val)
maxlen=max(maxlen1,maxlen2,maxlen3)
print(maxlen)

15


In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train = pad_sequences(
    X_train,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_train = pad_sequences(
    Y_train,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)
X_test = pad_sequences(
    X_test,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_test = pad_sequences(
    Y_test,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)
X_val = pad_sequences(
    X_val,
    maxlen=maxlen,
    padding="post",
    value=word2idx["<PAD>"]
)
Y_val = pad_sequences(
    Y_val,
    maxlen=maxlen,
    padding="post",
    value=tag2idx["<pad>"]
)

In [12]:
from torch.utils.data import Dataset, DataLoader
class Calendardataset(Dataset):
  def __init__(self,X,Y):
    self.X=X
    self.Y=Y
  def __len__(self):
    return len(self.X)
  def __getitem__(self,idx):
    return torch.LongTensor(self.X[idx]),torch.LongTensor(self.Y[idx])


In [13]:
train_dataset = Calendardataset(X_train, Y_train)
val_dataset = Calendardataset(X_val, Y_val)
test_dataset = Calendardataset(X_test, Y_test)

In [14]:
batch_size=32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [15]:
for X_batch, Y_batch in train_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break
for X_batch, Y_batch in val_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break
for X_batch, Y_batch in test_dataloader:
    print(X_batch.shape)
    print(Y_batch.shape)
    break

torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])
torch.Size([32, 15])


In [16]:
# ============================================================
# TASK-3
# YEAR → MONTH → DAY → HOUR → MINUTE
# ============================================================

from torch.nn.utils.rnn import pad_sequence


# ============================================================
# 1. YEAR VOCABULARY
# ============================================================

year2idx = {"<NA>": 0}

for sample in train_data:

    target = sample["target_string"].split("|")[2]

    if target != "NA":

        parts = target.split(" ")

        if "-" in parts[0]:

            year = int(parts[0].split("-")[0])

            if year not in year2idx:
                year2idx[year] = len(year2idx)


idx2year = {
    idx: year
    for year, idx in year2idx.items()
}


print("Year vocabulary:")
print(year2idx)


# ============================================================
# 2. PREPARE TASK-3 DATA
# ============================================================

def prepare_task3(data):

    X = []
    Y = []

    for sample in data:

        # -----------------------------
        # INPUT
        # -----------------------------

        sentence = [
            word2idx.get(
                word,
                word2idx["<UNK>"]
            )
            for word in sample["tokens"]
        ]


        # -----------------------------
        # TARGET
        #
        # [YEAR, MONTH, DAY, HOUR, MINUTE]
        #
        # 0 = NA
        # -----------------------------

        year = 0
        month = 0
        day = 0
        hour = 0
        minute = 0

        target = sample["target_string"].split("|")[2]

        if target != "NA":

            parts = target.split(" ")


            # -------------------------
            # DATE
            # -------------------------

            if "-" in parts[0]:

                date_parts = parts[0].split("-")

                year_value = int(date_parts[0])
                month_value = int(date_parts[1])
                day_value = int(date_parts[2])

                year = year2idx.get(
                    year_value,
                    0
                )

                month = month_value

                day = day_value


            # -------------------------
            # TIME
            # -------------------------

            if len(parts) >= 2 and ":" in parts[1]:

                time_parts = parts[1].split(":")

                hour = int(time_parts[0]) + 1
                minute = int(time_parts[1]) + 1


            # -------------------------
            # TIME ONLY
            # -------------------------

            elif ":" in parts[0]:

                time_parts = parts[0].split(":")

                hour = int(time_parts[0]) + 1
                minute = int(time_parts[1]) + 1


        X.append(
            torch.tensor(
                sentence,
                dtype=torch.long
            )
        )

        Y.append(
            torch.tensor(
                [
                    year,
                    month,
                    day,
                    hour,
                    minute
                ],
                dtype=torch.long
            )
        )

    return X, Y


X3_train, Y3_train = prepare_task3(train_data)
X3_val, Y3_val = prepare_task3(val_data)
X3_test, Y3_test = prepare_task3(test_data)


print(
    "\nNumber of training examples:",
    len(X3_train)
)

print(
    "Number of validation examples:",
    len(X3_val)
)

print(
    "Number of test examples:",
    len(X3_test)
)


# ============================================================
# 3. DATASET
# ============================================================

class Task3Dataset(Dataset):

    def __init__(self, X, Y):

        self.X = X
        self.Y = Y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.Y[idx]


train_dataset_3 = Task3Dataset(
    X3_train,
    Y3_train
)

val_dataset_3 = Task3Dataset(
    X3_val,
    Y3_val
)

test_dataset_3 = Task3Dataset(
    X3_test,
    Y3_test
)


# ============================================================
# 4. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    X_batch, Y_batch = zip(*batch)

    X_batch = pad_sequence(
        X_batch,
        batch_first=True,
        padding_value=word2idx["<PAD>"]
    )

    # Y is already fixed length = 5
    Y_batch = torch.stack(Y_batch)

    return X_batch, Y_batch


# ============================================================
# 5. DATALOADERS
# ============================================================

batch_size = 32

train_dataloader_3 = DataLoader(
    train_dataset_3,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_dataloader_3 = DataLoader(
    val_dataset_3,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

test_dataloader_3 = DataLoader(
    test_dataset_3,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)


# ============================================================
# 6. ENCODER-DECODER MODEL
# ============================================================

class Seq2Seq(nn.Module):

    def __init__(
        self,
        embedding_layer,
        embedding_dim=300,
        hidden_dim=512
    ):

        super().__init__()


        # --------------------------------
        # Encoder
        # --------------------------------

        self.embedding = embedding_layer

        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


        # --------------------------------
        # Decoder
        # --------------------------------

        self.decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


        # --------------------------------
        # Initial decoder input
        # --------------------------------

        self.sos_embedding = nn.Parameter(
            torch.randn(1, embedding_dim)
        )


        # --------------------------------
        # Embeddings for previous
        # decoder outputs
        # --------------------------------

        self.year_embedding = nn.Embedding(
            len(year2idx),
            embedding_dim
        )

        self.month_embedding = nn.Embedding(
            13,
            embedding_dim
        )

        self.day_embedding = nn.Embedding(
            32,
            embedding_dim
        )

        self.hour_embedding = nn.Embedding(
            25,
            embedding_dim
        )

        self.minute_embedding = nn.Embedding(
            61,
            embedding_dim
        )


        # --------------------------------
        # Output heads
        # --------------------------------

        self.year_fc = nn.Linear(
            hidden_dim,
            len(year2idx)
        )

        self.month_fc = nn.Linear(
            hidden_dim,
            13
        )

        self.day_fc = nn.Linear(
            hidden_dim,
            32
        )

        self.hour_fc = nn.Linear(
            hidden_dim,
            25
        )

        self.minute_fc = nn.Linear(
            hidden_dim,
            61
        )


    def forward(
        self,
        src,
        target=None,
        teacher_forcing=True
    ):

        batch_size = src.size(0)


        # ====================================================
        # ENCODER
        # ====================================================

        embedded_src = self.embedding(src)

        _, (hidden, cell) = self.encoder(
            embedded_src
        )


        # ====================================================
        # DECODER
        # ====================================================

        outputs = []


        # First decoder input = SOS

        decoder_input = self.sos_embedding.unsqueeze(0).repeat(
            batch_size,
            1,
            1
        )


        # ====================================================
        # FIVE DECODER STEPS
        # ====================================================

        for step in range(5):

            decoder_output, (hidden, cell) = self.decoder(
                decoder_input,
                (hidden, cell)
            )


            # [batch, 1, hidden_dim]
            decoder_output = decoder_output[:, 0, :]


            # ------------------------------------------------
            # Select appropriate output head
            # ------------------------------------------------

            if step == 0:

                output = self.year_fc(
                    decoder_output
                )

            elif step == 1:

                output = self.month_fc(
                    decoder_output
                )

            elif step == 2:

                output = self.day_fc(
                    decoder_output
                )

            elif step == 3:

                output = self.hour_fc(
                    decoder_output
                )

            else:

                output = self.minute_fc(
                    decoder_output
                )


            outputs.append(output)


            # =================================================
            # Prepare input for NEXT decoder step
            # =================================================

            if step < 4:

                if (
                    teacher_forcing
                    and target is not None
                ):

                    # Use actual previous component

                    next_value = target[:, step]


                else:

                    # Use model's prediction

                    next_value = output.argmax(
                        dim=-1
                    )


                # --------------------------------------------
                # Convert previous component into embedding
                # --------------------------------------------

                if step == 0:

                    decoder_input = self.year_embedding(
                        next_value
                    )

                elif step == 1:

                    decoder_input = self.month_embedding(
                        next_value
                    )

                elif step == 2:

                    decoder_input = self.day_embedding(
                        next_value
                    )

                elif step == 3:

                    decoder_input = self.hour_embedding(
                        next_value
                    )


                decoder_input = decoder_input.unsqueeze(1)


        return outputs


# ============================================================
# 7. CREATE MODEL
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model3 = Seq2Seq(
    embedding_layer=embedding,
    embedding_dim=300,
    hidden_dim=512
)


model3 = model3.to(device)


print("\nDevice:", device)
print(model3)


# ============================================================
# 8. LOSS + OPTIMIZER
# ============================================================

criterion3 = nn.CrossEntropyLoss()


optimizer3 = torch.optim.Adam(
    model3.parameters(),
    lr=0.001,
    weight_decay=1e-5
)


# ============================================================
# 9. TRAINING
# ============================================================

num_epochs = 100
patience = 15

best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses_3 = []
val_losses_3 = []


for epoch in range(num_epochs):

    # ========================================================
    # TRAIN
    # ========================================================

    model3.train()

    total_train_loss = 0


    for X_batch, Y_batch in train_dataloader_3:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)


        outputs = model3(
            X_batch,
            Y_batch,
            teacher_forcing=True
        )


        # --------------------------------
        # Five losses
        # --------------------------------

        year_loss = criterion3(
            outputs[0],
            Y_batch[:, 0]
        )

        month_loss = criterion3(
            outputs[1],
            Y_batch[:, 1]
        )

        day_loss = criterion3(
            outputs[2],
            Y_batch[:, 2]
        )

        hour_loss = criterion3(
            outputs[3],
            Y_batch[:, 3]
        )

        minute_loss = criterion3(
            outputs[4],
            Y_batch[:, 4]
        )


        loss = (
            year_loss
            + month_loss
            + day_loss
            + hour_loss
            + minute_loss
        )


        optimizer3.zero_grad()

        loss.backward()

        optimizer3.step()


        total_train_loss += loss.item()


    train_loss = (
        total_train_loss /
        len(train_dataloader_3)
    )

    train_losses_3.append(
        train_loss
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model3.eval()

    total_val_loss = 0


    with torch.no_grad():

        for X_batch, Y_batch in val_dataloader_3:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)


            outputs = model3(
                X_batch,
                Y_batch,
                teacher_forcing=True
            )


            year_loss = criterion3(
                outputs[0],
                Y_batch[:, 0]
            )

            month_loss = criterion3(
                outputs[1],
                Y_batch[:, 1]
            )

            day_loss = criterion3(
                outputs[2],
                Y_batch[:, 2]
            )

            hour_loss = criterion3(
                outputs[3],
                Y_batch[:, 3]
            )

            minute_loss = criterion3(
                outputs[4],
                Y_batch[:, 4]
            )


            loss = (
                year_loss
                + month_loss
                + day_loss
                + hour_loss
                + minute_loss
            )


            total_val_loss += loss.item()


    val_loss = (
        total_val_loss /
        len(val_dataloader_3)
    )

    val_losses_3.append(
        val_loss
    )


    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        best_model_state = {
            key: value.cpu().clone()
            for key, value in model3.state_dict().items()
        }

        best_epoch = epoch + 1

        print(
            f"  → New best model! "
            f"Val Loss: {best_val_loss:.4f}"
        )


    else:

        epochs_without_improvement += 1

        print(
            f"  → No improvement "
            f"({epochs_without_improvement}/{patience})"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= patience:

        print(
            f"\nEarly stopping triggered."
            f"\nBest epoch: {best_epoch}"
            f"\nBest validation loss: "
            f"{best_val_loss:.4f}"
        )

        break


# ============================================================
# 10. RESTORE BEST MODEL
# ============================================================

model3.load_state_dict(
    best_model_state
)

model3.to(device)


print(
    f"\nBest model restored from "
    f"epoch {best_epoch} "
    f"with validation loss "
    f"{best_val_loss:.4f}"
)


torch.save(
    model3.state_dict(),
    "task3_best_model.pt"
)


print("Best Task-3 model saved.")




Year vocabulary:
{'<NA>': 0, 2026: 1, 2027: 2}

Number of training examples: 3381
Number of validation examples: 724
Number of test examples: 725

Device: cuda
Seq2Seq(
  (embedding): Embedding(649, 300)
  (encoder): LSTM(300, 512, batch_first=True)
  (decoder): LSTM(300, 512, batch_first=True)
  (year_embedding): Embedding(3, 300)
  (month_embedding): Embedding(13, 300)
  (day_embedding): Embedding(32, 300)
  (hour_embedding): Embedding(25, 300)
  (minute_embedding): Embedding(61, 300)
  (year_fc): Linear(in_features=512, out_features=3, bias=True)
  (month_fc): Linear(in_features=512, out_features=13, bias=True)
  (day_fc): Linear(in_features=512, out_features=32, bias=True)
  (hour_fc): Linear(in_features=512, out_features=25, bias=True)
  (minute_fc): Linear(in_features=512, out_features=61, bias=True)
)
Epoch 1/100 | Train Loss: 5.8233 | Val Loss: 5.0291
  → New best model! Val Loss: 5.0291
Epoch 2/100 | Train Loss: 4.7463 | Val Loss: 4.5573
  → New best model! Val Loss: 4.5573
Ep

In [17]:
# ============================================================
# TASK-3 EVALUATION
# Piece-wise + Overall Exact Match Accuracy
# ============================================================

def evaluate_task3():

    model3.eval()

    total = 0

    correct_year = 0
    correct_month = 0
    correct_day = 0
    correct_hour = 0
    correct_minute = 0

    overall_correct = 0


    with torch.no_grad():

        for X_batch, Y_batch in test_dataloader_3:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)


            # ---------------------------------------------
            # Run encoder-decoder
            # ---------------------------------------------

            outputs = model3(
                X_batch,
                teacher_forcing=False
            )


            # ---------------------------------------------
            # Predictions for each component
            # ---------------------------------------------

            pred_year = outputs[0].argmax(dim=1)
            pred_month = outputs[1].argmax(dim=1)
            pred_day = outputs[2].argmax(dim=1)
            pred_hour = outputs[3].argmax(dim=1)
            pred_minute = outputs[4].argmax(dim=1)


            # ---------------------------------------------
            # Individual accuracies
            # ---------------------------------------------

            correct_year += (
                (pred_year == Y_batch[:, 0])
                .sum()
                .item()
            )

            correct_month += (
                (pred_month == Y_batch[:, 1])
                .sum()
                .item()
            )

            correct_day += (
                (pred_day == Y_batch[:, 2])
                .sum()
                .item()
            )

            correct_hour += (
                (pred_hour == Y_batch[:, 3])
                .sum()
                .item()
            )

            correct_minute += (
                (pred_minute == Y_batch[:, 4])
                .sum()
                .item()
            )


            # ---------------------------------------------
            # Overall exact match
            # ---------------------------------------------

            all_correct = (
                (pred_year == Y_batch[:, 0])
                &
                (pred_month == Y_batch[:, 1])
                &
                (pred_day == Y_batch[:, 2])
                &
                (pred_hour == Y_batch[:, 3])
                &
                (pred_minute == Y_batch[:, 4])
            )

            overall_correct += (
                all_correct
                .sum()
                .item()
            )


            total += Y_batch.size(0)


    # ========================================================
    # CALCULATE ACCURACIES
    # ========================================================

    year_accuracy = (
        correct_year / total * 100
    )

    month_accuracy = (
        correct_month / total * 100
    )

    day_accuracy = (
        correct_day / total * 100
    )

    hour_accuracy = (
        correct_hour / total * 100
    )

    minute_accuracy = (
        correct_minute / total * 100
    )

    overall_accuracy = (
        overall_correct / total * 100
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n")
    print("=" * 60)
    print("TASK-3 EVALUATION")
    print("=" * 60)

    print(
        f"Overall Exact-Match Accuracy: "
        f"{overall_accuracy:.2f}%"
    )

    print("-" * 60)
    print("Piece-wise Accuracy:")
    print("-" * 60)

    print(
        f"YEAR   | Correct: {correct_year:4d} / {total:4d} "
        f"| Accuracy: {year_accuracy:.2f}%"
    )

    print(
        f"MONTH  | Correct: {correct_month:4d} / {total:4d} "
        f"| Accuracy: {month_accuracy:.2f}%"
    )

    print(
        f"DAY    | Correct: {correct_day:4d} / {total:4d} "
        f"| Accuracy: {day_accuracy:.2f}%"
    )

    print(
        f"HOUR   | Correct: {correct_hour:4d} / {total:4d} "
        f"| Accuracy: {hour_accuracy:.2f}%"
    )

    print(
        f"MINUTE | Correct: {correct_minute:4d} / {total:4d} "
        f"| Accuracy: {minute_accuracy:.2f}%"
    )

    print("=" * 60)


# ============================================================
# RUN EVALUATION
# ============================================================

evaluate_task3()



TASK-3 EVALUATION
Overall Exact-Match Accuracy: 90.62%
------------------------------------------------------------
Piece-wise Accuracy:
------------------------------------------------------------
YEAR   | Correct:  710 /  725 | Accuracy: 97.93%
MONTH  | Correct:  692 /  725 | Accuracy: 95.45%
DAY    | Correct:  678 /  725 | Accuracy: 93.52%
HOUR   | Correct:  711 /  725 | Accuracy: 98.07%
MINUTE | Correct:  715 /  725 | Accuracy: 98.62%


In [18]:
# ============================================================
# 1. SET SAVE LOCATION
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/Calendar-Assistant NLU"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Save directory:")
print(SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory:
/content/drive/MyDrive/Calendar-Assistant NLU


In [19]:
# ============================================================
# 2. SAVE COMPLETE TASK-3 CHECKPOINT
# ============================================================

task3_checkpoint = {

    # ========================================================
    # MAIN MODEL
    # ========================================================

    "model_state_dict": model3.state_dict(),


    # ========================================================
    # WORD VOCABULARY
    # ========================================================

    "word2idx": word2idx,
    "idx2word": idx2word,


    # ========================================================
    # ENCODER EMBEDDING
    # ========================================================

    "embedding_state_dict": embedding.state_dict(),


    # ========================================================
    # TASK-3 YEAR VOCABULARY
    # ========================================================

    "year2idx": year2idx,
    "idx2year": idx2year,


    # ========================================================
    # MODEL CONFIGURATION
    # ========================================================

    "embedding_dim": 300,
    "hidden_dim": 512,


    # ========================================================
    # TRAINING INFORMATION
    # ========================================================

    "best_val_loss": best_val_loss,
    "best_epoch": best_epoch
}


task3_path = os.path.join(
    SAVE_DIR,
    "task3_final.pt"
)


torch.save(
    task3_checkpoint,
    task3_path
)


print("Task-3 checkpoint saved successfully.")
print()
print("Location:")
print(task3_path)

Task-3 checkpoint saved successfully.

Location:
/content/drive/MyDrive/Calendar-Assistant NLU/task3_final.pt


In [20]:
# ============================================================
# 3. VERIFY TASK-3 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    task3_path,
    map_location="cpu"
)

print("Task-3 checkpoint loaded successfully.\n")

print("Contents:")
print("-" * 50)

for key in checkpoint.keys():
    print(key)

print("-" * 50)

print("\nFile location:")
print(task3_path)

print("\nFile exists:")
print(os.path.exists(task3_path))

print("\nBest epoch:")
print(checkpoint["best_epoch"])

print("\nBest validation loss:")
print(checkpoint["best_val_loss"])

Task-3 checkpoint loaded successfully.

Contents:
--------------------------------------------------
model_state_dict
word2idx
idx2word
embedding_state_dict
year2idx
idx2year
embedding_dim
hidden_dim
best_val_loss
best_epoch
--------------------------------------------------

File location:
/content/drive/MyDrive/Calendar-Assistant NLU/task3_final.pt

File exists:
True

Best epoch:
39

Best validation loss:
0.9356192943194638
